## Setup

In [4]:
from dotenv import load_dotenv
from utils import chat_interface
import os

In [5]:
load_dotenv()

True

## Run

In [ ]:
# TODO: Develop your agents under `agentic/agents`
# TODO: Develop your tools under `agentic/tools`
# TODO: Modify `agentic/workflow` in order to orchestrate your agents

In [8]:
# IDEALLY YOUR ONLY IMPORT HERE IS:
# from agentic.workflow import orchestrator

from agentic.workflow import orchestrator

In [ ]:
await chat_interface(orchestrator, "1")

User: Hi
Assistant: Hello! How can I assist you today?
User: q
Assistant: Goodbye!


In [5]:
list(orchestrator.get_state_history(
    config = {
        "configurable": {
            "thread_id": "1",
        }
    }
))[0].values["messages"]

[HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}, id='43c56100-7a4e-4ff0-adb7-1fbbfdac82e3'),
 AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 60, 'total_tokens': 70, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-C1loRWd5jRqktu5Fut6YZAZsTdc6S', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--0ab48c4a-6130-4dc9-96cd-271e2be7b7c8-0', usage_metadata={'input_tokens': 60, 'output_tokens': 10, 'total_tokens': 70, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]

## Routing Tests (Sample Tickets)

In [ ]:
from uuid import uuid4
from langchain_core.messages import HumanMessage

sample_tickets = [
    {
        "name": "KB hit -> resolver",
        "latest_message": "How do I cancel my reservation?",
        "channel": "chat",
        "main_issue_type": "reservation",
        "tags": "reservation, cancellation",
    },
    {
        "name": "KB miss -> escalation",
        "latest_message": "ZXQ-91 edge-protocol mismatch in latent socket handshake",
        "channel": "chat",
        "main_issue_type": "general",
        "tags": "unknown",
    },
    {
        "name": "Sensitive type -> escalation",
        "latest_message": "There is a fraudulent charge on my account and I need urgent help.",
        "channel": "email",
        "main_issue_type": "fraud",
        "tags": "fraud, billing",
    },
]


def build_ticket_state(sample: dict, thread_id: str) -> dict:
    return {
        "ticket": {
            "ticket_id": thread_id,
            "account_id": "cultpass",
            "user_id": "demo-user",
            "external_user_id": "demo-external-user",
            "channel": sample["channel"],
            "status": "open",
            "main_issue_type": sample.get("main_issue_type"),
            "tags": sample.get("tags"),
            "latest_message": sample["latest_message"],
        },
        "messages": [HumanMessage(content=sample["latest_message"])],
        "short_term_memory": {},
        "long_term_memory": {},
    }


async def demo_routing(orchestrator):
    print("=== Knowledge Retrieval + Escalation Demonstration ===")

    for sample in sample_tickets:
        thread_id = str(uuid4())
        state_input = build_ticket_state(sample, thread_id)
        config = {"configurable": {"thread_id": thread_id}}

        result = await orchestrator.ainvoke(input=state_input, config=config)
        resolution = result.get("resolution") or {}
        classification = result.get("classification") or {}
        retrieved_context = result.get("retrieved_context") or []

        # Drive workflow to terminal state if needed.
        while not resolution and result.get("next_agent") in {"resolver", "escalation", "classifier"}:
            result = await orchestrator.ainvoke(input={"messages": []}, config=config)
            resolution = result.get("resolution") or resolution
            classification = result.get("classification") or classification
            retrieved_context = result.get("retrieved_context") or retrieved_context

        kb_hits = 0
        if retrieved_context and isinstance(retrieved_context[0], dict):
            kb_hits = int(retrieved_context[0].get("total_found", 0) or 0)

        final_route = "escalation" if resolution.get("resolved") is False else "resolver/end"

        print(f"\nCase: {sample['name']}")
        print(f"  Message         : {sample['latest_message']}")
        print(f"  Classified as   : {classification.get('issue_type', 'unknown')}")
        print(f"  Class confidence: {classification.get('confidence', 'n/a')}")
        print(f"  KB hits         : {kb_hits}")
        print(f"  Final route     : {final_route}")
        print(f"  Resolved        : {resolution.get('resolved', 'n/a')}")
        print(f"  Action          : {resolution.get('action_taken', 'n/a')}")


await demo_routing(orchestrator)

## Tool Usage Tests (Direct Sample Operations)

In [ ]:
# Tool Usage Tests (Direct Sample Operations)
import json
from agentic.tools.crm_tools import lookup_customer, lookup_reservation, issue_refund

# Use a real sample user from external dataset for repeatable demos.
with open("data/external/cultpass_users.jsonl", "r", encoding="utf-8") as f:
    sample_user = json.loads(next(f))

external_user_id = sample_user["id"]
print(f"Using sample external_user_id: {external_user_id}")

customer_result = await lookup_customer(external_user_id=external_user_id)
print("\nlookup_customer result:")
print(json.dumps(customer_result, indent=2))

reservation_result = await lookup_reservation(external_user_id=external_user_id)
print("\nlookup_reservation result:")
print(json.dumps(reservation_result, indent=2))

# Optional refund demo if a confirmed reservation exists.
confirmed = [
    r for r in reservation_result.get("reservations", [])
    if r.get("status") == "confirmed"
]

if confirmed:
    refund_result = await issue_refund(
        external_user_id=external_user_id,
        reservation_id=confirmed[0]["reservation_id"],
        reason="Rubric demonstration: test refund workflow",
    )
    print("\nissue_refund result:")
    print(json.dumps(refund_result, indent=2))
else:
    print("\nissue_refund skipped: no confirmed reservations for this sample user.")

## Persistent Memory & Personalization Demo

In [ ]:
import json
from agentic.tools.ticket_tools import get_long_term_memory, update_long_term_memory

print("=== Persistent Customer Memory Demonstration ===\n")

# Use a consistent customer ID for this demo
demo_customer_id = "demo-returning-customer-001"

print(f"Customer ID: {demo_customer_id}\n")

# STEP 1: Retrieve initial memory (first visit)
print("--- STEP 1: Initial Memory Retrieval (First Visit) ---")
initial_memory = await get_long_term_memory(demo_customer_id)
print(f"Memory on first visit:")
print(json.dumps(initial_memory, indent=2))

# STEP 2: Simulate a ticket resolution
print("\n--- STEP 2: Ticket Resolved & Memory Updated ---")
resolution_summary = "Customer resolved reservation cancellation issue. Preferences: prefers email communication, morning availability."
print(f"Resolution: {resolution_summary}")

update_result = await update_long_term_memory(
    external_user_id=demo_customer_id,
    resolution_summary=resolution_summary,
    issue_type="reservation",
    preferences={
        "preferred_contact": "email",
        "best_time": "morning",
        "resolution_speed_preference": "quick"
    }
)
print(f"\nMemory update result: {update_result['message']}")

# STEP 3: Add more interactions to show cumulative history
print("\n--- STEP 3: Additional Interactions & Cumulative Memory ---")
await update_long_term_memory(
    external_user_id=demo_customer_id,
    resolution_summary="Customer inquired about billing dispute. Resolved with refund of $50.",
    issue_type="billing",
    preferences={"repeat_issue_risk": "billing"}
)
print("Added second resolution to memory.")

# STEP 4: Retrieve memory again (showing persistence and context)
print("\n--- STEP 4: Memory Retrieval on Return Visit ---")
updated_memory = await get_long_term_memory(demo_customer_id)
print(f"Memory on return visit (now contains history):")
print(json.dumps(updated_memory, indent=2))

# STEP 5: Show how memory enables personalization
print("\n--- STEP 5: Personalization Benefits ---")
print(f"✓ Past Issues Known: {updated_memory['past_issue_types']}")
print(f"✓ Customer Preferences Retrieved: {updated_memory['preferences']}")
print(f"✓ Resolution History: {len(updated_memory['past_resolutions'])} previous resolutions")
print("\nWith this context, the resolver agent can now:")
print("  • Use preferred communication method (email)")
print("  • Be aware of customer's history with billing issues")
print("  • Personalize responses based on preferences")
print("  • Avoid repeating previous solutions")